<div align="center">

<!-- MOTIONSALT branded banner. Rendered as HTML for the logo mark + gradient. -->
<div style="background:linear-gradient(135deg,#0f172a 0%,#1e1b4b 60%,#312e81 100%);padding:28px 24px;border-radius:14px;color:#f8fafc;font-family:-apple-system,Segoe UI,Roboto,sans-serif;">
  <div style="display:flex;align-items:center;justify-content:center;gap:14px;">
    <div style="width:44px;height:44px;border-radius:10px;background:linear-gradient(135deg,#22d3ee,#a855f7);display:flex;align-items:center;justify-content:center;font-weight:900;font-size:22px;color:#0f172a;">M</div>
    <div style="font-size:30px;font-weight:800;letter-spacing:2px;">MOTIONSALT</div>
  </div>
  <div style="margin-top:8px;font-size:14px;opacity:0.85;letter-spacing:3px;text-transform:uppercase;">Anime&nbsp;Video&nbsp;Upscaler</div>
  <div style="margin-top:14px;font-size:14px;opacity:0.75;max-width:640px;margin-left:auto;margin-right:auto;">A free, no-install, GPU-in-the-cloud alternative to Topaz Video AI. Powered by AnimeJaNai&nbsp;V3 and Real-ESRGAN AnimeVideo&nbsp;v3.</div>
  <div style="margin-top:18px;font-size:12px;opacity:0.7;">
    <a style="color:#a5f3fc;text-decoration:none;" href="https://github.com/YOUR-USERNAME/motionsalt-upscaler">github.com/YOUR-USERNAME/motionsalt-upscaler</a>
  </div>
</div>

</div>

---

**How this works:** step through the four cells below in order. Each cell is a self-contained step of a wizard — Connect ➜ Upload ➜ Configure ➜ Download. You never need to read or edit any code.

## Step 1 — Connect

Click **Connect** below. This verifies your GPU, installs the dependencies, and downloads the AI model weights from the MOTIONSALT GitHub Releases (never from HuggingFace — see the README for why).

In [ ]:
#@title 🔌 Step 1 — Connect { display-mode: "form" }
#@markdown Click the **Connect** button that appears below this cell after you run it.
import os, sys, subprocess, shutil, json, time, urllib.request, urllib.error
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ---------- MOTIONSALT global state ----------
MS = globals().setdefault("MOTIONSALT", {})
MS.setdefault("workdir", Path("/content/motionsalt"))
MS["workdir"].mkdir(parents=True, exist_ok=True)
MS.setdefault("weights_dir", MS["workdir"] / "weights")
MS["weights_dir"].mkdir(parents=True, exist_ok=True)
MS.setdefault("connected", False)

# ---------- Config: where to pull weights from ----------
# The Colab notebook ONLY ever downloads weights from this GitHub repo's Releases.
# The HuggingFace upstream is mirrored by a scheduled Action; the notebook itself
# never contacts HuggingFace directly.
GH_REPO   = "YOUR-USERNAME/motionsalt-upscaler"          # <-- change to your fork
GH_TAG    = "latest"                                    # "latest" resolves to the newest weights-vX.Y.Z tag
WEIGHTS = {
    "LOW":    "2x_AnimeJaNaiV3_SuperUltraCompact.pth",
    "MEDIUM": "2x_AnimeJaNaiV3_UltraCompact.pth",
    "HIGH":   "realesr-animevideov3.pth",
}
MS["weights_map"] = WEIGHTS
MS["gh_repo"]     = GH_REPO

# ---------- Branded status log ----------
_log = widgets.HTML(value="")
_lines = []
def status(kind, msg):
    icon = {"ok":"✅","warn":"⚠️","err":"❌","run":"⏳","info":"•"}[kind]
    color = {"ok":"#16a34a","warn":"#d97706","err":"#dc2626","run":"#0891b2","info":"#475569"}[kind]
    _lines.append(f'<div style="font-family:ui-monospace,Menlo,monospace;font-size:12.5px;color:{color};padding:2px 0;">{icon}&nbsp;&nbsp;{msg}</div>')
    _log.value = "".join(_lines)

def section(title):
    _lines.append(f'<div style="margin:10px 0 4px 0;font-weight:600;color:#0f172a;font-family:-apple-system,Segoe UI,sans-serif;">{title}</div>')
    _log.value = "".join(_lines)

# ---------- The Connect button ----------
btn = widgets.Button(
    description="Connect",
    icon="plug",
    button_style="primary",
    layout=widgets.Layout(width="180px", height="42px"),
)
badge = widgets.HTML(
    '<span style="display:inline-block;padding:4px 10px;border-radius:999px;background:#e2e8f0;color:#334155;font-size:11px;font-family:-apple-system,sans-serif;">not connected</span>'
)
header = widgets.HTML(
    '<div style="font-family:-apple-system,Segoe UI,sans-serif;font-weight:700;font-size:16px;color:#0f172a;">Connect to a MOTIONSALT session</div>'
    '<div style="font-family:-apple-system,Segoe UI,sans-serif;font-size:12.5px;color:#475569;margin-bottom:8px;">Verifies GPU · installs dependencies · pulls model weights from GitHub Releases</div>'
)

def _run(cmd, quiet=True):
    """Run a shell command; return (returncode, tail_of_output)."""
    p = subprocess.run(cmd, shell=isinstance(cmd,str), capture_output=True, text=True)
    if not quiet and p.returncode != 0:
        print(p.stdout[-2000:]); print(p.stderr[-2000:])
    return p.returncode, (p.stderr or p.stdout)[-400:]

def _resolve_release_tag():
    """Ask GitHub for the newest release tag (unauthenticated is fine — public repo)."""
    if GH_TAG != "latest":
        return GH_TAG
    url = f"https://api.github.com/repos/{GH_REPO}/releases/latest"
    with urllib.request.urlopen(url, timeout=20) as r:
        data = json.loads(r.read().decode("utf-8"))
    return data["tag_name"]

def _download(url, dest: Path, label: str):
    """Streaming download with a live progress bar."""
    bar = widgets.IntProgress(value=0, min=0, max=100, description=label,
                              layout=widgets.Layout(width="100%"),
                              bar_style="info")
    pct = widgets.HTML(value="0%")
    row = widgets.HBox([bar, pct])
    display(row)
    req = urllib.request.Request(url, headers={"User-Agent":"motionsalt-upscaler"})
    with urllib.request.urlopen(req, timeout=60) as r:
        total = int(r.headers.get("Content-Length", "0")) or 0
        read = 0
        with dest.open("wb") as f:
            while True:
                chunk = r.read(1 << 20)
                if not chunk:
                    break
                f.write(chunk); read += len(chunk)
                if total:
                    p = int(read * 100 / total)
                    bar.value = p; pct.value = f"{p}%"
        if total:
            bar.value = 100; pct.value = "100%"
        bar.bar_style = "success"

def on_connect(_):
    btn.disabled = True
    badge.value = '<span style="display:inline-block;padding:4px 10px;border-radius:999px;background:#fef3c7;color:#92400e;font-size:11px;">connecting…</span>'
    _lines.clear(); _log.value = ""

    # 1. GPU
    section("1 / 4 · GPU")
    try:
        import torch
        if not torch.cuda.is_available():
            status("err", "No CUDA GPU detected. Enable GPU: Runtime → Change runtime type → GPU.")
            badge.value = '<span style="padding:4px 10px;border-radius:999px;background:#fee2e2;color:#991b1b;font-size:11px;">no GPU</span>'
            btn.disabled = False
            return
        name = torch.cuda.get_device_name(0)
        status("ok", f"GPU detected: <b>{name}</b>")
    except Exception as e:
        status("err", f"PyTorch not importable yet: {e}")

    # 2. System deps (ffmpeg)
    section("2 / 4 · System dependencies")
    if shutil.which("ffmpeg"):
        status("ok", "ffmpeg already present.")
    else:
        status("run", "Installing ffmpeg…")
        rc, tail = _run("apt-get -qq update && apt-get -qq install -y ffmpeg")
        status("ok" if rc==0 else "err", "ffmpeg installed." if rc==0 else f"ffmpeg install failed: {tail}")

    # 3. Python deps
    section("3 / 4 · Python packages")
    pkgs = ["opencv-python-headless", "numpy", "spandrel", "tqdm"]
    # spandrel: universal loader that supports both AnimeJaNai (compact/SRVGGNet) and Real-ESRGAN (SRVGGNet) checkpoints.
    status("run", "Installing " + ", ".join(pkgs) + " …")
    rc, tail = _run([sys.executable, "-m", "pip", "install", "-q", *pkgs])
    status("ok" if rc==0 else "err", "Python packages ready." if rc==0 else f"pip failed: {tail}")

    # 4. Weights — from THIS GitHub repo, never HuggingFace
    section("4 / 4 · Model weights (from GitHub Releases)")
    try:
        tag = _resolve_release_tag()
        status("ok", f"Resolved release tag: <b>{tag}</b>")
    except Exception as e:
        status("err", f"Could not reach GitHub API: {e}")
        btn.disabled = False; return

    ok_all = True
    for tier, fname in WEIGHTS.items():
        dest = MS["weights_dir"] / fname
        if dest.exists() and dest.stat().st_size > 0:
            status("ok", f"{tier} — {fname} already cached.")
            continue
        url = f"https://github.com/{GH_REPO}/releases/download/{tag}/{fname}"
        status("run", f"{tier} — downloading {fname}…")
        try:
            _download(url, dest, tier)
            status("ok", f"{tier} — downloaded.")
        except Exception as e:
            status("err", f"{tier} — download failed: {e}")
            ok_all = False

    if ok_all:
        MS["connected"] = True
        MS["release_tag"] = tag
        badge.value = '<span style="padding:4px 10px;border-radius:999px;background:#dcfce7;color:#166534;font-size:11px;">connected</span>'
        status("ok", "<b>Ready.</b> Continue to Step 2.")
    else:
        badge.value = '<span style="padding:4px 10px;border-radius:999px;background:#fee2e2;color:#991b1b;font-size:11px;">error</span>'

    btn.disabled = False

btn.on_click(on_connect)

panel = widgets.VBox([
    header,
    widgets.HBox([btn, badge]),
    _log,
], layout=widgets.Layout(border="1px solid #e2e8f0", padding="14px", border_radius="10px"))
display(panel)


## Step 2 — Upload your video

Pick a video file from your device. It's copied into the Colab VM only — nothing is sent to a third-party service.

In [ ]:
#@title 📤 Step 2 — Upload video { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from pathlib import Path
import shutil, os

MS = globals().setdefault("MOTIONSALT", {})
if not MS.get("connected"):
    display(HTML('<div style="padding:10px;background:#fee2e2;color:#991b1b;border-radius:8px;font-family:sans-serif;">⚠️ Run Step 1 first (Connect).</div>'))
else:
    header = widgets.HTML(
        '<div style="font-family:-apple-system,Segoe UI,sans-serif;font-weight:700;font-size:16px;color:#0f172a;">Upload the video you want to upscale</div>'
        '<div style="font-family:-apple-system,Segoe UI,sans-serif;font-size:12.5px;color:#475569;margin-bottom:10px;">MP4, MKV, MOV, WebM, AVI… anything ffmpeg reads.</div>'
    )
    pick_btn = widgets.Button(description="Choose file…", icon="upload",
                              button_style="primary",
                              layout=widgets.Layout(width="180px", height="42px"))
    prog = widgets.IntProgress(value=0, min=0, max=100, description="Copy",
                               layout=widgets.Layout(width="100%"),
                               bar_style="info")
    prog_pct = widgets.HTML("0%")
    prog_row = widgets.HBox([prog, prog_pct])
    prog_row.layout.display = "none"
    result = widgets.HTML("")

    def on_pick(_):
        # Use the Colab file uploader; then copy into workspace with progress.
        from google.colab import files
        pick_btn.disabled = True
        result.value = '<div style="font-family:sans-serif;font-size:12.5px;color:#475569;">Waiting for browser file picker…</div>'
        uploaded = files.upload()
        if not uploaded:
            result.value = '<div style="padding:8px;background:#fef3c7;color:#92400e;border-radius:6px;font-family:sans-serif;">No file selected.</div>'
            pick_btn.disabled = False
            return
        name, data = next(iter(uploaded.items()))
        src_tmp = Path("/content") / name
        # google.colab.files.upload() already wrote it into /content; we just move it.
        dest = MS["workdir"] / "input" / name
        dest.parent.mkdir(parents=True, exist_ok=True)
        prog_row.layout.display = "flex"
        # Copy with progress (chunked, so the bar is real, not fake).
        total = len(data); read = 0
        with open(src_tmp, "rb") as fin, open(dest, "wb") as fout:
            while True:
                chunk = fin.read(1 << 20)
                if not chunk: break
                fout.write(chunk); read += len(chunk)
                prog.value = int(read * 100 / max(total, 1))
                prog_pct.value = f"{prog.value}%"
        prog.value = 100; prog_pct.value = "100%"; prog.bar_style = "success"
        try: os.remove(src_tmp)
        except OSError: pass
        MS["input_path"] = dest
        size_mb = dest.stat().st_size / 1e6
        result.value = (
            f'<div style="padding:10px;background:#dcfce7;color:#166534;border-radius:8px;font-family:sans-serif;">'
            f'✅ Uploaded <b>{name}</b> ({size_mb:.1f} MB). Continue to Step 3.'
            f'</div>'
        )
        pick_btn.disabled = False

    pick_btn.on_click(on_pick)
    display(widgets.VBox([header, pick_btn, prog_row, result],
                         layout=widgets.Layout(border="1px solid #e2e8f0", padding="14px", border_radius="10px")))


## Step 3 — Configure & process

Pick a quality tier and dial in the filters. Then click **Start Processing**. A progress bar shows real frame-by-frame progress.

In [ ]:
#@title ⚙️ Step 3 — Configure & process { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, HTML
from pathlib import Path
import subprocess, shutil, math, json, time, os

MS = globals().setdefault("MOTIONSALT", {})
if not MS.get("connected"):
    display(HTML('<div style="padding:10px;background:#fee2e2;color:#991b1b;border-radius:8px;font-family:sans-serif;">⚠️ Run Step 1 first (Connect).</div>'))
elif not MS.get("input_path"):
    display(HTML('<div style="padding:10px;background:#fee2e2;color:#991b1b;border-radius:8px;font-family:sans-serif;">⚠️ Run Step 2 first (Upload).</div>'))
else:
    # -------- Widgets --------
    tier = widgets.Dropdown(
        options=[
            ("LOW  — AnimeJaNai V3 SuperUltraCompact 2× (fastest)", "LOW"),
            ("MEDIUM — AnimeJaNai V3 UltraCompact 2× (balanced)", "MEDIUM"),
            ("HIGH — Real-ESRGAN AnimeVideo v3 2× (best quality)", "HIGH"),
        ],
        value="MEDIUM",
        description="Quality",
        style={"description_width": "160px"},
        layout=widgets.Layout(width="640px"),
    )

    def slider(desc, default=0):
        return widgets.IntSlider(
            value=default, min=0, max=100, step=1, description=desc,
            style={"description_width": "160px"},
            layout=widgets.Layout(width="640px"),
            continuous_update=False,
        )

    s_revert    = slider("Revert Compression", 0)
    s_detail    = slider("Improve Detail", 0)
    s_sharpen   = slider("Sharpen", 15)
    s_denoise   = slider("Reduce Noise", 0)
    s_dehalo    = slider("Dehalo", 0)
    s_deblur    = slider("Anti-alias/Deblur", 0)
    s_recover   = slider("Recover Original Detail", 0)
    cb_1080     = widgets.Checkbox(value=False, description="Downscale to 1080p height (preserve aspect ratio)",
                                   indent=False)

    start = widgets.Button(description="Start Processing", icon="play",
                           button_style="success",
                           layout=widgets.Layout(width="220px", height="44px"))
    frame_prog = widgets.IntProgress(value=0, min=0, max=100, description="Frames",
                                     layout=widgets.Layout(width="100%"), bar_style="info")
    frame_pct  = widgets.HTML("0%")
    stage_lbl  = widgets.HTML("")
    log        = widgets.HTML("")
    _msgs = []
    def logline(kind, msg):
        icon = {"ok":"✅","warn":"⚠️","err":"❌","run":"⏳","info":"•"}[kind]
        color = {"ok":"#16a34a","warn":"#d97706","err":"#dc2626","run":"#0891b2","info":"#475569"}[kind]
        _msgs.append(f'<div style="font-family:ui-monospace,Menlo,monospace;font-size:12.5px;color:{color};padding:2px 0;">{icon}&nbsp;&nbsp;{msg}</div>')
        log.value = "".join(_msgs)

    hdr = widgets.HTML(
        '<div style="font-family:-apple-system,Segoe UI,sans-serif;font-weight:700;font-size:16px;color:#0f172a;">Configure processing</div>'
        '<div style="font-family:-apple-system,Segoe UI,sans-serif;font-size:12.5px;color:#475569;margin-bottom:10px;">Defaults are safe. All sliders are 0–100.</div>'
    )

    def start_processing(_):
        start.disabled = True
        _msgs.clear(); log.value = ""
        try:
            _do_process()
        except Exception as e:
            logline("err", f"Processing failed: {e}")
            raise
        finally:
            start.disabled = False

    # ---------- The actual pipeline ----------
    def _ffprobe(path, args):
        out = subprocess.check_output(["ffprobe","-v","error", *args, str(path)]).decode().strip()
        return out

    def _load_model(tier_val):
        """Load the checkpoint for the chosen tier via spandrel, which supports
        both AnimeJaNai (SRVGGNetCompact variants) and Real-ESRGAN AnimeVideo v3."""
        import torch
        from spandrel import ModelLoader
        weight_file = MS["weights_map"][tier_val]
        path = MS["weights_dir"] / weight_file
        model = ModelLoader().load_from_file(str(path))
        model.cuda().eval()
        return model

    def _apply_filters(bgr, params):
        """Non-AI cleanup passes. Each slider is implemented as a real, distinct
        step. Strength 0 disables that step entirely (no-op)."""
        import cv2, numpy as np
        out = bgr

        # (a) Revert Compression — deblocking pre-pass (bilateral: preserves edges,
        # smooths flat blocky regions typical of low-bitrate H.264).
        if params["revert"] > 0:
            k = params["revert"] / 100.0
            d = int(3 + 6 * k)                     # neighborhood size
            sC = 20 + 60 * k; sS = 20 + 40 * k
            out = cv2.bilateralFilter(out, d, sC, sS)

        # (b) Reduce Noise — fastNlMeans (color-aware, edge preserving).
        if params["denoise"] > 0:
            h = 3 + 12 * (params["denoise"] / 100.0)
            out = cv2.fastNlMeansDenoisingColored(out, None, h, h, 7, 21)

        # (c) Dehalo — edge-aware guided-style pass. We detect high-freq edges,
        # dilate slightly to get the halo band, and pull those pixels toward the
        # median-filtered version. This is the standard dehalo idea.
        if params["dehalo"] > 0:
            k = params["dehalo"] / 100.0
            gray = cv2.cvtColor(out, cv2.COLOR_BGR2GRAY)
            edges = cv2.Canny(gray, 60, 180)
            band  = cv2.dilate(edges, np.ones((3,3), np.uint8), iterations=1)
            band  = cv2.subtract(band, edges)      # the "halo ring", not the edge itself
            med   = cv2.medianBlur(out, 5)
            mask  = (band > 0)[..., None].astype(np.float32) * (0.35 + 0.55 * k)
            out   = (out.astype(np.float32) * (1 - mask) + med.astype(np.float32) * mask).astype(np.uint8)

        # (d) Anti-alias / Deblur — mild Gaussian to break up staircase aliasing
        # on diagonals introduced by the upscale, then unsharp with small radius
        # to keep line definition. Net effect: smoother diagonals, same acuity.
        if params["deblur"] > 0:
            k = params["deblur"] / 100.0
            sig = 0.4 + 0.9 * k
            blur = cv2.GaussianBlur(out, (0,0), sig)
            out  = cv2.addWeighted(out, 1.0 + 0.35 * k, blur, -0.35 * k, 0)

        # (e) Improve Detail — CLAHE on the L channel. Restores local micro-
        # contrast the model may have flattened.
        if params["detail"] > 0:
            k = params["detail"] / 100.0
            lab = cv2.cvtColor(out, cv2.COLOR_BGR2LAB)
            L, A, B = cv2.split(lab)
            clahe = cv2.createCLAHE(clipLimit=1.0 + 2.5 * k, tileGridSize=(8,8))
            L = clahe.apply(L)
            out = cv2.cvtColor(cv2.merge([L,A,B]), cv2.COLOR_LAB2BGR)

        # (f) Sharpen — unsharp mask with a larger radius than the deblur step.
        if params["sharpen"] > 0:
            k = params["sharpen"] / 100.0
            blur = cv2.GaussianBlur(out, (0,0), 1.2)
            amt  = 0.2 + 1.2 * k
            out  = cv2.addWeighted(out, 1.0 + amt, blur, -amt, 0)

        return out

    def _recover_blend(upscaled_bgr, original_bgr, strength_0_100):
        """Recover Original Detail — resize the ORIGINAL frame up to the
        upscaled size and blend a fraction of it back in. At strength=0 the
        model output is used as-is; at strength=100 half of the signal is the
        (bilinearly upscaled) original — the classic 'keep source flavor' knob.
        """
        if strength_0_100 <= 0:
            return upscaled_bgr
        import cv2
        h, w = upscaled_bgr.shape[:2]
        naive = cv2.resize(original_bgr, (w, h), interpolation=cv2.INTER_LANCZOS4)
        alpha = 0.5 * (strength_0_100 / 100.0)   # up to 50% source contribution
        import numpy as np
        return (upscaled_bgr.astype(np.float32) * (1 - alpha) + naive.astype(np.float32) * alpha).astype(np.uint8)

    def _infer_tensor(model, bgr):
        """Run one BGR uint8 frame through the loaded model; return BGR uint8."""
        import torch, numpy as np
        rgb = bgr[:, :, ::-1].copy()
        t = torch.from_numpy(rgb).float().div_(255.0).permute(2,0,1).unsqueeze_(0).cuda()
        with torch.no_grad():
            y = model(t)
        y = y.clamp_(0,1).squeeze_(0).permute(1,2,0).mul_(255.0).round_().byte().cpu().numpy()
        return y[:, :, ::-1].copy()

    def _do_process():
        import cv2, numpy as np, torch
        in_path = MS["input_path"]
        out_dir = MS["workdir"] / "out"; out_dir.mkdir(exist_ok=True)
        stem = in_path.stem

        # 1. Probe input
        stage_lbl.value = "<b>Reading source metadata…</b>"
        fps = _ffprobe(in_path, ["-select_streams","v:0","-show_entries","stream=r_frame_rate","-of","csv=p=0"])
        num, den = (fps.split("/") + ["1"])[:2]
        fps_f = float(num) / float(den) if float(den) else 30.0
        nframes = int(_ffprobe(in_path, ["-select_streams","v:0","-count_packets","-show_entries","stream=nb_read_packets","-of","csv=p=0"]) or "0")
        logline("ok", f"Source: {fps_f:.3f} fps · ~{nframes or 'unknown'} frames.")

        # 2. Load model
        stage_lbl.value = "<b>Loading model…</b>"
        logline("run", f"Loading {tier.value} model…")
        model = _load_model(tier.value)
        logline("ok", f"{tier.value} model loaded on GPU.")

        params = dict(
            revert=s_revert.value, detail=s_detail.value, sharpen=s_sharpen.value,
            denoise=s_denoise.value, dehalo=s_dehalo.value, deblur=s_deblur.value,
            recover=s_recover.value,
        )

        # 3. Open source with OpenCV, pipe upscaled frames straight to ffmpeg's
        #    stdin. Avoids writing millions of PNGs to disk on the free tier.
        cap = cv2.VideoCapture(str(in_path))
        if not cap.isOpened():
            raise RuntimeError("OpenCV could not open the input video.")
        w  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        out_w, out_h = w * 2, h * 2      # every supported model is 2× only.

        video_only = out_dir / f"{stem}_upscaled_noaudio.mp4"
        ff = subprocess.Popen(
            ["ffmpeg","-y","-f","rawvideo","-pix_fmt","bgr24",
             "-s",f"{out_w}x{out_h}","-r",f"{fps_f}",
             "-i","-",
             "-c:v","libx264","-preset","medium","-crf","16","-pix_fmt","yuv420p",
             str(video_only)],
            stdin=subprocess.PIPE, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )

        frame_prog.max = max(nframes, 1)
        stage_lbl.value = "<b>Upscaling frames…</b>"
        i = 0; t0 = time.time()
        while True:
            ok, frame = cap.read()
            if not ok: break
            up = _infer_tensor(model, frame)
            up = _recover_blend(up, frame, params["recover"])
            up = _apply_filters(up, params)
            ff.stdin.write(up.tobytes())
            i += 1
            if i % 4 == 0 or i == nframes:
                frame_prog.value = min(i, frame_prog.max)
                elapsed = time.time() - t0
                fps_now = i / max(elapsed, 1e-6)
                frame_pct.value = f"{i}/{nframes or '?'} · {fps_now:.2f} fps"
        cap.release()
        try: ff.stdin.close()
        except Exception: pass
        ff.wait()
        logline("ok", f"Upscaled {i} frames in {time.time()-t0:.1f}s.")

        # 4. Mux original audio back in (preserve audio — never drop it).
        stage_lbl.value = "<b>Muxing original audio…</b>"
        with_audio = out_dir / f"{stem}_upscaled.mp4"
        rc = subprocess.run(
            ["ffmpeg","-y","-i",str(video_only),"-i",str(in_path),
             "-map","0:v:0","-map","1:a:0?","-c:v","copy","-c:a","aac","-b:a","192k",
             "-shortest", str(with_audio)],
            capture_output=True, text=True,
        )
        if rc.returncode != 0:
            # No audio in source, or codec mismatch — keep the silent output.
            shutil.move(str(video_only), str(with_audio))
            logline("warn","Source had no audio track; output is silent.")
        else:
            try: os.remove(video_only)
            except OSError: pass
            logline("ok","Audio muxed back in.")

        final = with_audio

        # 5. Optional 1080p-height downscale — aspect-ratio-preserving, NOT a
        #    forced 1920x1080 canvas. Height=1080; width scales with source AR.
        if cb_1080.value:
            stage_lbl.value = "<b>Downscaling to 1080p height (aspect-preserving)…</b>"
            down = out_dir / f"{stem}_upscaled_1080p.mp4"
            # scale=-2:1080  ->  h=1080, w = round-to-even to keep AR
            rc = subprocess.run(
                ["ffmpeg","-y","-i",str(final),
                 "-vf","scale=-2:1080:flags=lanczos",
                 "-c:v","libx264","-preset","medium","-crf","17","-pix_fmt","yuv420p",
                 "-c:a","copy",
                 str(down)],
                capture_output=True, text=True,
            )
            if rc.returncode == 0:
                final = down
                logline("ok","Downscaled to 1080p height, aspect ratio preserved.")
            else:
                logline("warn","1080p downscale failed — keeping full-res output.")

        MS["output_path"] = final
        stage_lbl.value = f"<b>Done.</b> Output: <code>{final.name}</code>"
        logline("ok", f"Ready for Step 4. File: <b>{final.name}</b> · {final.stat().st_size/1e6:.1f} MB.")

    start.on_click(start_processing)

    display(widgets.VBox([
        hdr, tier,
        s_revert, s_detail, s_sharpen, s_denoise, s_dehalo, s_deblur, s_recover,
        cb_1080,
        widgets.HTML("<hr style='border:none;border-top:1px solid #e2e8f0;margin:8px 0;'>"),
        start,
        stage_lbl,
        widgets.HBox([frame_prog, frame_pct]),
        log,
    ], layout=widgets.Layout(border="1px solid #e2e8f0", padding="14px", border_radius="10px")))


## Step 4 — Download

Click the button. Your browser downloads the result directly. No public/shareable link is generated — the file only exists on your machine and the Colab VM.

In [ ]:
#@title ⬇️ Step 4 — Download result { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, HTML

MS = globals().setdefault("MOTIONSALT", {})
if not MS.get("output_path"):
    display(HTML('<div style="padding:10px;background:#fee2e2;color:#991b1b;border-radius:8px;font-family:sans-serif;">⚠️ Run Step 3 first (Configure & process).</div>'))
else:
    out = MS["output_path"]
    hdr = widgets.HTML(
        f'<div style="font-family:-apple-system,Segoe UI,sans-serif;font-weight:700;font-size:16px;color:#0f172a;">Your file is ready</div>'
        f'<div style="font-family:-apple-system,Segoe UI,sans-serif;font-size:12.5px;color:#475569;margin-bottom:10px;"><code>{out.name}</code> · {out.stat().st_size/1e6:.1f} MB</div>'
    )
    btn = widgets.Button(description="⬇️ Download Result", button_style="primary",
                         layout=widgets.Layout(width="240px", height="46px"))
    status = widgets.HTML("")

    def on_click(_):
        from google.colab import files
        btn.disabled = True
        status.value = '<div style="font-family:sans-serif;color:#475569;font-size:12.5px;">Preparing browser download…</div>'
        files.download(str(out))
        status.value = '<div style="padding:10px;background:#dcfce7;color:#166534;border-radius:8px;font-family:sans-serif;">✅ Download started in your browser.</div>'
        btn.disabled = False

    btn.on_click(on_click)
    display(widgets.VBox([hdr, btn, status],
                         layout=widgets.Layout(border="1px solid #e2e8f0", padding="14px", border_radius="10px")))


---

<div align="center" style="font-family:-apple-system,Segoe UI,sans-serif;color:#64748b;font-size:12px;padding:10px;">
MOTIONSALT Upscaler · MIT-licensed wrapper · powered by
<a href="https://github.com/the-database/mpv-upscale-2x_animejanai">AnimeJaNai V3</a> and
<a href="https://github.com/xinntao/Real-ESRGAN">Real-ESRGAN AnimeVideo v3</a>.<br>
Source: <a href="https://github.com/YOUR-USERNAME/motionsalt-upscaler">github.com/YOUR-USERNAME/motionsalt-upscaler</a>
</div>